# Social Network – Data Cleaning & Recommendations
Using a small, messy social-network dataset (`social_data.json`) we will:

1. **Clean and structure the data**
2. **Find "People You May Know"** – friend suggestions based on mutual friends
3. **Find "Pages You Might Like"** – page suggestions based on shared interests

In [ ]:
import json
from collections import defaultdict, Counter
import pandas as pd

---
# Part 1 – Cleaning and Structuring the Data

## 1.1 Load the raw data

In [ ]:
with open('social_data.json') as f:
    data = json.load(f)

users, pages = data['users'], data['pages']
print(f'{len(users)} users, {len(pages)} pages loaded')
pd.DataFrame(users).head(10)

## 1.2 Inspect the problems

In [ ]:
ids = [str(u['id']) for u in users]
print('Duplicate user IDs  :', [i for i, c in Counter(ids).items() if c > 1])
print('Users with no name  :', [u['id'] for u in users if not u['name'].strip()])
print('Users with no activity:', [u['id'] for u in users if not u['friends'] and not u['liked_pages']])
print('IDs stored as text  :', [u['id'] for u in users if isinstance(u['id'], str)])
print('Self-friendships    :', [u['id'] for u in users if u['id'] in u['friends']])
print('Duplicate page IDs  :', [i for i, c in Counter(p['id'] for p in pages).items() if c > 1])

## 1.3 Clean the users
- Convert all IDs to integers
- Trim and title-case names; drop users with no name
- Merge duplicate user records
- Remove duplicate friends / pages

In [ ]:
clean_users = {}
for u in users:
    uid = int(u['id'])
    name = u['name'].strip().title()
    if not name:
        continue                                   # drop nameless users
    friends = {int(f) for f in u['friends']}
    liked = {int(p) for p in u['liked_pages']}
    if uid in clean_users:                         # merge duplicates
        clean_users[uid]['friends'] |= friends
        clean_users[uid]['liked_pages'] |= liked
    else:
        clean_users[uid] = {'id': uid, 'name': name, 'friends': friends, 'liked_pages': liked}

print(f'{len(clean_users)} users after cleaning')

## 1.4 Clean the pages
Trim names, drop blank names, keep one name per page ID.

In [ ]:
clean_pages = {}
for p in pages:
    name = p['name'].strip()
    if name and p['id'] not in clean_pages:
        clean_pages[p['id']] = name.title() if name.isupper() or name.islower() else name

print(f'{len(clean_pages)} pages after cleaning')
clean_pages

## 1.5 Fix the relationships
- Remove self-friendships and friends who don't exist
- Make friendships **two-way** (if A is B's friend, B is A's friend)
- Remove likes for pages that don't exist
- Remove inactive users (no friends and no likes)

In [ ]:
for uid, u in clean_users.items():
    u['friends'] = {f for f in u['friends'] if f != uid and f in clean_users}
    u['liked_pages'] = {p for p in u['liked_pages'] if p in clean_pages}

for uid, u in clean_users.items():               # make friendships mutual
    for f in u['friends']:
        clean_users[f]['friends'].add(uid)

inactive = [uid for uid, u in clean_users.items() if not u['friends'] and not u['liked_pages']]
for uid in inactive:
    del clean_users[uid]
print('Removed inactive users:', inactive)
print(f'Final: {len(clean_users)} users')

## 1.6 Structure the data
Build quick-lookup tables and tidy DataFrames.

In [ ]:
friends_of = {uid: u['friends'] for uid, u in clean_users.items()}
likes_of   = {uid: u['liked_pages'] for uid, u in clean_users.items()}
name_of    = {uid: u['name'] for uid, u in clean_users.items()}

users_df = pd.DataFrame([{'id': uid, 'name': u['name'],
                          'num_friends': len(u['friends']),
                          'num_pages_liked': len(u['liked_pages'])}
                         for uid, u in sorted(clean_users.items())])

friendships_df = pd.DataFrame(sorted({tuple(sorted((a, b))) for a in friends_of for b in friends_of[a]}),
                              columns=['user_1', 'user_2'])

likes_df = pd.DataFrame([(uid, pid, clean_pages[pid]) for uid in likes_of for pid in sorted(likes_of[uid])],
                        columns=['user_id', 'page_id', 'page_name'])

print(f'{len(friendships_df)} friendships, {len(likes_df)} likes')
users_df.head(10)

In [ ]:
# Most popular pages
likes_df['page_name'].value_counts().rename('likes').to_frame()

In [ ]:
# Save the cleaned data
clean_json = {'users': [{**u, 'friends': sorted(u['friends']), 'liked_pages': sorted(u['liked_pages'])}
                        for u in clean_users.values()],
              'pages': [{'id': k, 'name': v} for k, v in clean_pages.items()]}
with open('clean_social_data.json', 'w') as f:
    json.dump(clean_json, f, indent=2)
print('Saved clean_social_data.json')

---
# Part 2 – Finding "People You May Know"
**Idea:** if you and someone else share many friends, you probably know each other.

For a user, look at their friends' friends. Skip the user and people they already know. Count how many mutual friends each candidate has, then rank by that count.

In [ ]:
def people_you_may_know(user_id, top_n=5):
    mutual = Counter()
    for friend in friends_of[user_id]:
        for fof in friends_of[friend]:
            if fof != user_id and fof not in friends_of[user_id]:
                mutual[fof] += 1
    return mutual.most_common(top_n)

user = 1
print(f"People {name_of[user]} may know:")
for uid, count in people_you_may_know(user):
    print(f"  {name_of[uid]:<10} – {count} mutual friend(s)")

In [ ]:
# Show who the mutual friends are
def mutual_friends(a, b):
    return sorted(name_of[f] for f in friends_of[a] & friends_of[b])

for uid, _ in people_you_may_know(user, 3):
    print(f"{name_of[user]} & {name_of[uid]}: {mutual_friends(user, uid)}")

In [ ]:
# Top suggestion for every user
pymk = [{'user': name_of[uid],
         'suggestions': ', '.join(f"{name_of[s]} ({c})" for s, c in people_you_may_know(uid, 3)) or '—'}
        for uid in sorted(friends_of)]
pymk_df = pd.DataFrame(pymk)
pymk_df

---
# Part 3 – Finding "Pages You Might Like"
**Idea:** people who like the same pages as you have similar interests, so you may like other pages they like.

1. Find users who share at least one liked page with you.
2. Give each one a **similarity score**: the number of pages you both like.
3. Add that score to every page they like that you haven't liked yet.
4. Recommend the pages with the highest totals.

In [ ]:
def pages_you_might_like(user_id, top_n=5):
    my_pages = likes_of[user_id]
    scores = Counter()
    for other, their_pages in likes_of.items():
        if other == user_id:
            continue
        shared = len(my_pages & their_pages)
        if shared == 0:
            continue
        for page in their_pages - my_pages:
            scores[page] += shared
    return scores.most_common(top_n)

print(f"{name_of[user]} already likes: {[clean_pages[p] for p in sorted(likes_of[user])]}\n")
print(f"Pages {name_of[user]} might like:")
for pid, score in pages_you_might_like(user):
    print(f"  {clean_pages[pid]:<22} – score {score}")

In [ ]:
# Bonus: also boost pages your friends like (social signal)
def pages_you_might_like_v2(user_id, top_n=5, friend_weight=1):
    scores = Counter(dict(pages_you_might_like(user_id, top_n=None)))
    for friend in friends_of[user_id]:
        for page in likes_of[friend] - likes_of[user_id]:
            scores[page] += friend_weight
    return scores.most_common(top_n)

print(f"Pages {name_of[user]} might like (interests + friends):")
for pid, score in pages_you_might_like_v2(user):
    print(f"  {clean_pages[pid]:<22} – score {score}")

In [ ]:
# Top page suggestions for every user
pyml_df = pd.DataFrame([{'user': name_of[uid],
                         'suggestions': ', '.join(clean_pages[p] for p, _ in pages_you_might_like_v2(uid, 3)) or '—'}
                        for uid in sorted(likes_of)])
pyml_df

In [ ]:
# Export all recommendations
recs = pymk_df.rename(columns={'suggestions': 'people_you_may_know'}).merge(
       pyml_df.rename(columns={'suggestions': 'pages_you_might_like'}), on='user')
recs.to_csv('recommendations.csv', index=False)
print('Saved recommendations.csv')
recs.head()